# 02 — Logistic Regression（11 個差值特徵）

| 項目 | 設定 |
|------|------|
| 特徵集 | 11 個 `_diff` 差值特徵（消除共線性） |
| 標準化 | StandardScaler（封裝於 Pipeline，fit train only） |
| VIF 說明 | 原始特徵與差值特徵完全線性相依（diff = home − away），故直接選 diff 特徵取代 VIF 迭代篩選 |
| 缺失處理 | SimpleImputer(median)，僅 fit train |
| 輸出 | lr_predictions.csv、lr_metrics.csv、lr_coefficients.csv |

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, roc_auc_score,
                              f1_score, brier_score_loss,
                              classification_report, confusion_matrix)

PROJECT_ROOT = next(
    path for path in [Path('..').resolve(), Path('.').resolve()]
    if (path / 'data' / 'processed' / 'model_ready_games.csv').exists()
)
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'model_ready_games.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
METRICS_DIR = OUTPUT_DIR / 'metrics'
PREDICTIONS_DIR = OUTPUT_DIR / 'predictions'
for path in [METRICS_DIR, PREDICTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DIFF_COLS = [
    'win_rate_diff', 'runs_scored_diff', 'run_diff_10',
    'starter_ERA_diff', 'starter_WHIP_diff', 'starter_FIP_diff',
    'lineup_OPS_diff', 'lineup_OBP_diff', 'lineup_SLG_diff',
    'bullpen_ERA_diff', 'bullpen_WHIP_diff',
]

## 1. 讀取資料與切分

In [2]:
df = pd.read_csv(DATA_PATH, dtype={'year': str})
df = df[df['year'] != '2026'].copy()  # 排除 2026

LABEL = 'home_win'

train = df[df['year'].isin([str(y) for y in range(2018, 2025)])].copy()
test  = df[df['year'] == '2025'].copy()

X_train, y_train = train[DIFF_COLS], train[LABEL].astype(int)
X_test,  y_test  = test[DIFF_COLS],  test[LABEL].astype(int)

print(f'訓練集：{len(train)} 場（2018–2024）')
print(f'測試集：{len(test)} 場（2025）')
print(f'使用特徵：{len(DIFF_COLS)} 個（僅 _diff）')

訓練集：1947 場（2018–2024）
測試集：358 場（2025）
使用特徵：11 個（僅 _diff）


## 2. 前處理：SimpleImputer + StandardScaler

兩者均只 `fit` 訓練集，再 `transform` 測試集，防止資料洩漏。

In [3]:
imputer      = SimpleImputer(strategy='median')
X_train_imp  = imputer.fit_transform(X_train)
X_test_imp   = imputer.transform(X_test)

print(f'train 缺失率：{X_train.isna().mean().mean():.2%}')
print(f'test  缺失率：{X_test.isna().mean().mean():.2%}')

train 缺失率：3.01%
test  缺失率：1.90%


## 3. 訓練 Logistic Regression（含 StandardScaler Pipeline）

In [4]:
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr',     LogisticRegression(C=0.1, max_iter=1000, random_state=42)),
])
lr_pipe.fit(X_train_imp, y_train)
print('訓練完成')

訓練完成


## 4. 評估

In [5]:
y_pred = lr_pipe.predict(X_test_imp)
y_prob = lr_pipe.predict_proba(X_test_imp)[:, 1]

metrics = {
    'Model':       'Logistic Regression',
    'Feature_Set': 'Diff 11',
    'Accuracy':    round(accuracy_score(y_test, y_pred), 4),
    'AUC':         round(roc_auc_score(y_test, y_prob), 4),
    'F1':          round(f1_score(y_test, y_pred), 4),
    'Brier_Score': round(brier_score_loss(y_test, y_prob), 4),
}

for k, v in metrics.items():
    print(f'{k:<15}: {v}')

Model          : Logistic Regression
Feature_Set    : Diff 11
Accuracy       : 0.5335
AUC            : 0.538
F1             : 0.664
Brier_Score    : 0.2472


In [6]:
print(classification_report(y_test, y_pred, target_names=['客隊勝(0)', '主隊勝(1)']))

cm = confusion_matrix(y_test, y_pred)
pd.DataFrame(cm,
             index   = ['實際客勝', '實際主勝'],
             columns = ['預測客勝', '預測主勝'])

              precision    recall  f1-score   support

      客隊勝(0)       0.46      0.16      0.24       163
      主隊勝(1)       0.55      0.85      0.66       195

    accuracy                           0.53       358
   macro avg       0.51      0.50      0.45       358
weighted avg       0.51      0.53      0.47       358



,預測客勝,預測主勝
實際客勝,26,137
實際主勝,30,165


## 5. 係數解讀（標準化後）

In [7]:
coef = lr_pipe.named_steps['lr'].coef_[0]
coef_df = (pd.DataFrame({'feature': DIFF_COLS, 'coefficient': coef})
             .sort_values('coefficient', key=abs, ascending=False)
             .reset_index(drop=True))
print('係數（絕對值由大到小）：')
coef_df

係數（絕對值由大到小）：


,feature,coefficient
0,bullpen_WHIP_diff,-0.187049
1,run_diff_10,0.131010
2,bullpen_ERA_diff,0.109010
3,starter_FIP_diff,-0.084094
4,runs_scored_diff,-0.082880
5,lineup_SLG_diff,0.077134
6,win_rate_diff,-0.066512
7,starter_WHIP_diff,-0.060900
8,lineup_OPS_diff,0.050130
9,starter_ERA_diff,0.034098


## 6. 輸出 CSV

In [8]:
# predictions
pred_df = test[['game_id', 'date', 'home_team', 'away_team', LABEL]].copy()
pred_df['lr_pred']    = y_pred
pred_df['lr_prob']    = y_prob.round(4)
pred_df['lr_correct'] = (y_pred == y_test.values).astype(int)
pred_df.to_csv(PREDICTIONS_DIR / 'lr_predictions.csv', index=False, encoding='utf-8-sig')
print('lr_predictions.csv 已輸出')

# metrics
pd.DataFrame([metrics]).to_csv(METRICS_DIR / 'lr_metrics.csv', index=False)
print('lr_metrics.csv 已輸出')

# coefficients
coef_df.to_csv(METRICS_DIR / 'lr_coefficients.csv', index=False)
print('lr_coefficients.csv 已輸出')

lr_predictions.csv 已輸出
lr_metrics.csv 已輸出
lr_coefficients.csv 已輸出
